# Header reader bake-off: TrOCR-small vs PaddleOCR PP-OCRv5 rec
Kaggle settings: **Accelerator = GPU (T4/P100), Internet = ON**. Upload `kaggle_train.zip` as a dataset (or into /kaggle/working).

In [ ]:
!pip -q install sentencepiece
import glob, os
z = glob.glob('/kaggle/input/**/kaggle_train.zip', recursive=True) + glob.glob('/kaggle/working/kaggle_train.zip')
print(z)
os.makedirs('/kaggle/working/proj', exist_ok=True)
!cd /kaggle/working/proj && unzip -qo {z[0]}
%cd /kaggle/working/proj

## 1. TrOCR-small (register no. + name)
Same student split for everything. Prints train-vs-val gap each epoch, early-stops on val CER, scores TEST once.

In [ ]:
!python -u scripts/finetune_header.py --field reg  --model microsoft/trocr-small-handwritten --epochs 12 --bs 16 --lr 4e-5 --syn 6000 --save 2>&1 | grep -v -i 'warn\|Loading weights' | tee trocr_reg.log

In [ ]:
!python -u scripts/finetune_header.py --field name --model microsoft/trocr-small-handwritten --epochs 40 --bs 8 --lr 3e-5 --save 2>&1 | grep -v -i 'warn\|Loading weights' | tee trocr_name.log

## 2. PaddleOCR PP-OCRv5 mobile rec (fine-tune from pretrained)

In [ ]:
!pip -q install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/ 2>&1 | tail -1
!git clone -q --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git /kaggle/working/PaddleOCR
!cd /kaggle/working/PaddleOCR && pip -q install -r requirements.txt && pip -q install -e . 2>&1 | tail -1
!python scripts/prep_paddle_data.py --out /kaggle/working/paddle_data --syn 6000
!wget -q -P /kaggle/working/pretrained https://paddle-model-weights.bj.bcebos.com/pretrained/PP-OCRv5_mobile_rec_pretrained.pdparams
!ls -la /kaggle/working/pretrained

In [ ]:
import subprocess
def paddle_train(field, epochs, bs, lr):
    d = f'/kaggle/working/paddle_data/{field}'
    cfg = '/kaggle/working/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml'
    o = [f'Global.pretrained_model=/kaggle/working/pretrained/PP-OCRv5_mobile_rec_pretrained',
         f'Global.save_model_dir=/kaggle/working/paddle_out/{field}',
         f'Global.epoch_num={epochs}', 'Global.eval_batch_step=[0,50]', 'Global.save_epoch_step=5',
         'Global.use_gpu=True', 'Global.character_dict_path=/kaggle/working/PaddleOCR/ppocr/utils/dict/ppocrv5_dict.txt',
         f'Optimizer.lr.learning_rate={lr}', 'Optimizer.lr.warmup_epoch=1',
         f'Train.dataset.data_dir={d}', f'Train.dataset.label_file_list=[{d}/train.txt]',
         f'Train.loader.batch_size_per_card={bs}',
         f'Eval.dataset.data_dir={d}', f'Eval.dataset.label_file_list=[{d}/val.txt]',
         f'Eval.loader.batch_size_per_card={bs}']
    cmd = f'cd /kaggle/working/PaddleOCR && python tools/train.py -c {cfg} -o ' + ' '.join(o)
    print(cmd); subprocess.run(cmd, shell=True)
paddle_train('reg', 30, 64, 2e-4)

In [ ]:
paddle_train('name', 80, 32, 1e-4)

## 3. Score Paddle on the TEST students (scored once) and compare
Paddle's `best_accuracy` is chosen on val; this evaluates it on test.

In [ ]:
import subprocess
for field in ('reg', 'name'):
    d = f'/kaggle/working/paddle_data/{field}'
    cfg = '/kaggle/working/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml'
    cmd = (f'cd /kaggle/working/PaddleOCR && python tools/eval.py -c {cfg} -o '
           f'Global.checkpoints=/kaggle/working/paddle_out/{field}/best_accuracy '
           f'Global.character_dict_path=/kaggle/working/PaddleOCR/ppocr/utils/dict/ppocrv5_dict.txt '
           f'Eval.dataset.data_dir={d} Eval.dataset.label_file_list=[{d}/test.txt]')
    print('=== PADDLE TEST', field); subprocess.run(cmd, shell=True)
print('
=== TrOCR TEST (from logs)')
!grep -h "TEST\|FINAL\|VAL\]" trocr_reg.log trocr_name.log

## 4. Download results
Models: `/kaggle/working/proj/models/trocr_*_small`, `/kaggle/working/paddle_out/*/best_accuracy*`.

In [ ]:
!cd /kaggle/working && zip -qr results.zip proj/models paddle_out/*/best_accuracy* proj/trocr_*.log